In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df = df.drop("customerID", axis=1)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Churn"]
)

print("Training Data Shape:", train_df.shape)
print("Testing Data Shape:", test_df.shape)

Training Data Shape: (5634, 20)
Testing Data Shape: (1409, 20)


In [3]:
X_train = train_df.drop("Churn", axis=1)

y_train = train_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Training Features Shape:", X_train.shape)
print("Training Target Shape:", y_train.shape)

print("\nTarget Distribution:")
print(y_train.value_counts())

Training Features Shape: (5634, 19)
Training Target Shape: (5634,)

Target Distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64


In [4]:
def create_features(data):
    data = data.copy()

    # 1. Group customers based on tenure
    data["tenure_group"] = pd.cut(
        data["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=["0-12", "13-24", "25-48", "49-72"]
    )

    # 2. Calculate average monthly spending
    data["avg_monthly_spend"] = (
        data["TotalCharges"] /
        data["tenure"].replace(0, np.nan)
    )

    # 3. Count subscribed services
    service_columns = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    data["service_count"] = (
        data[service_columns].eq("Yes").sum(axis=1)
    )

    # 4. Identify long-term contract customers
    data["long_term_contract"] = (
        data["Contract"].isin(["One year", "Two year"]).astype(int)
    )

    return data


X_train = create_features(X_train)

print("New Training Features Shape:", X_train.shape)
print("\nNew Features:")
print(X_train[[
    "tenure_group",
    "avg_monthly_spend",
    "service_count",
    "long_term_contract"
]].head())

New Training Features Shape: (5634, 23)

New Features:
     tenure_group  avg_monthly_spend  service_count  long_term_contract
3738        25-48          48.618571              3                   0
3151        13-24          76.770000              2                   0
4860        13-24          45.411538              3                   1
3867        25-48          73.296154              5                   1
3810         0-12          44.550000              1                   0


In [5]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nTotal Features:", len(numerical_features) + len(categorical_features))

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'service_count', 'long_term_contract']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']

Total Features: 23


In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformers
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing Pipeline Created Successfully!")

Preprocessing Pipeline Created Successfully!


In [7]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Logistic Regression Pipeline Created Successfully!")

Logistic Regression Pipeline Created Successfully!


In [8]:
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

print("Random Forest Pipeline Created Successfully!")

Random Forest Pipeline Created Successfully!


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "roc_auc": "roc_auc",
    "f1": "f1"
}

print("5-Fold Stratified Cross-Validation Created Successfully!")

5-Fold Stratified Cross-Validation Created Successfully!


In [10]:
logistic_scores = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Logistic Regression Results")
print("Mean ROC-AUC:", logistic_scores["test_roc_auc"].mean())
print("Mean F1 Score:", logistic_scores["test_f1"].mean())

Logistic Regression Results
Mean ROC-AUC: 0.8459380341050029
Mean F1 Score: 0.5930103972311492


In [11]:
random_forest_scores = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Random Forest Results")
print("Mean ROC-AUC:", random_forest_scores["test_roc_auc"].mean())
print("Mean F1 Score:", random_forest_scores["test_f1"].mean())

Random Forest Results
Mean ROC-AUC: 0.8262822847465564
Mean F1 Score: 0.5609469961896629


In [12]:
results_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "CV ROC-AUC": [
        logistic_scores["test_roc_auc"].mean(),
        random_forest_scores["test_roc_auc"].mean()
    ],
    "CV F1": [
        logistic_scores["test_f1"].mean(),
        random_forest_scores["test_f1"].mean()
    ]
})

results_df = results_df.round(3)

results_df

,Model,CV ROC-AUC,CV F1
0,Logistic Regression,0.846,0.593
1,Random Forest,0.826,0.561


In [13]:
from sklearn.dummy import DummyClassifier

dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

dummy_scores = cross_validate(
    dummy_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Dummy Classifier Results")
print("Mean ROC-AUC:", dummy_scores["test_roc_auc"].mean())
print("Mean F1 Score:", dummy_scores["test_f1"].mean())

Dummy Classifier Results
Mean ROC-AUC: 0.5
Mean F1 Score: 0.0


In [14]:
dummy_result = pd.DataFrame({
    "Model": ["Dummy Classifier"],
    "CV ROC-AUC": [dummy_scores["test_roc_auc"].mean()],
    "CV F1": [dummy_scores["test_f1"].mean()]
})

results_df = pd.concat(
    [results_df, dummy_result],
    ignore_index=True
)

results_df = results_df.round(3)

results_df

,Model,CV ROC-AUC,CV F1
0,Logistic Regression,0.846,0.593
1,Random Forest,0.826,0.561
2,Dummy Classifier,0.500,0.000


In [15]:
import os

os.makedirs("../outputs", exist_ok=True)

results_df.to_csv(
    "../outputs/task2_baseline_results.csv",
    index=False
)

print("Results table saved successfully!")

Results table saved successfully!


Task 2 – Features and Baseline Models: Conclusion

- Created four new features: "tenure_group", "avg_monthly_spend", "service_count", and "long_term_contract".
- Built numerical and categorical preprocessing pipelines using "ColumnTransformer".
- Trained Logistic Regression and Random Forest baseline pipelines.
- Evaluated both models using 5-fold Stratified Cross-Validation.
- Compared model performance using ROC-AUC and F1-score.
- Added Dummy Classifier as a bonus baseline.
- Saved the model comparison results in "outputs/task2_baseline_results.csv".

Key Learning

Learned feature engineering, data preprocessing, machine learning pipelines, baseline model building, and cross-validation.

Important Note

The test dataset was kept separate during cross-validation and will be used only for final model evaluation.